# Install dependencies

In [ ]:
!pip install ultralytics roboflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 129.6 MB/s eta 0:00:00


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# Download dataset

In [ ]:
# FILL IN YOUR OWN ROBOFLOW API KEY HERE
ROBOFLOW_API_KEY = "MTAy92kWRnfB3LV0COz8"

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("nours-workspace-qpy1n").project("cv-project-59lsq")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to CV-Project-1 in yolov8:: 100%|██████████| 193/193 [00:00<00:00, 8444.79it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
import os
for root, dirs, files in os.walk(dataset.location):
    print(root)

/content/CV-Project-1
/content/CV-Project-1/train
/content/CV-Project-1/train/labels
/content/CV-Project-1/train/images


In [ ]:
import os
import shutil

# Create valid folders
os.makedirs("/content/CV-Project-1/valid/images", exist_ok=True)
os.makedirs("/content/CV-Project-1/valid/labels", exist_ok=True)

# Move 20% of train images to valid
images = os.listdir("/content/CV-Project-1/train/images")
valid_images = images[:int(len(images)*0.2)]

for img in valid_images:
    shutil.move(f"/content/CV-Project-1/train/images/{img}",
                f"/content/CV-Project-1/valid/images/{img}")
    label = img.replace(".jpg", ".txt")
    shutil.move(f"/content/CV-Project-1/train/labels/{label}",
                f"/content/CV-Project-1/valid/labels/{label}")

print(f"Train: {len(os.listdir('/content/CV-Project-1/train/images'))} images")
print(f"Valid: {len(os.listdir('/content/CV-Project-1/valid/images'))} images")

Train: 76 images
Valid: 19 images


# Train Model

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # nano = fast, good for small dataset

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="bowling_pin_detector",
    patience=10,
)

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/CV-Project-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bowling_pin_detector-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a0b740ae360>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

# Eval


In [ ]:
metrics = model.val()
print(metrics)

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 489.8±124.7 MB/s, size: 12.6 KB)
val: Scanning /content/CV-Project-1/valid/labels.cache... 19 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 19/19 5.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.9it/s 1.1s
                   all         19         84      0.965      0.951      0.954      0.531
            pin-fallen         11         16      0.936      0.917      0.918       0.43
          pin-standing         19         68      0.993      0.985      0.991      0.633
Speed: 6.1ms preprocess, 11.8ms inference, 0.0ms loss, 5.6ms postprocess per image
Results saved to /content/runs/detect/val
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultraly

# Save Model

In [ ]:
import shutil
shutil.copy(
    "/content/runs/detect/bowling_pin_detector-2/weights/best.pt",
    "/content/best.pt"
)
print("Model saved!")

Model saved!


# Inference

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()

Saving test2.MP4 to test2.MP4


In [ ]:
import os

# Check what was actually uploaded
print(os.listdir("/content/"))

['.config', 'test2.MP4', 'model.pt', 'sample_data']


In [ ]:
video_path = "/content/test2.MP4"
print(f"Video ready: {video_path}")

Video ready: /content/test2.MP4


In [ ]:
model = YOLO("/content/model.pt")

cap            = cv2.VideoCapture(video_path)
fps            = cap.get(cv2.CAP_PROP_FPS)
width          = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height         = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
total_duration = total_frames / fps
cap.release()

print(f"FPS: {fps:.2f} | {width}x{height} | {total_frames} frames | {total_duration:.1f}s")

FPS: 29.97 | 464x848 | 769 frames | 25.7s


In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
NUM_PINS            = 5
CLASS_FALLEN        = 0      # check data.yaml — 0 = pin-fallen
CLASS_STANDING      = 1      # 1 = pin-standing
CONF_THRESHOLD      = 0.35   # YOLO detection confidence threshold
MAX_MATCH_DIST      = 120    # max centroid distance (px) to match detection to pin
INIT_STABLE_FRAMES  = 3      # consecutive clean frames required before initializing
CONSEC_THRESH       = 2      # consecutive fallen detections to confirm a fast fall
WINDOW_SIZE         = 6      # sliding window size for slow-fall ratio check
WINDOW_RATIO        = 0.70   # fraction of window that must be fallen to confirm
FALL_MISS_TOLERANCE = 5      # missed frames before clearing fall evidence window
GONE_CONFIRM_FRAMES = 15     # missed frames before assuming pin fell and slid (~0.5s)

# ── HELPERS ───────────────────────────────────────────────────────────────────

def get_center(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2)

def compute_iou(boxA, boxB):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    inter  = max(0, xB - xA) * max(0, yB - yA)
    union  = (boxA[2]-boxA[0])*(boxA[3]-boxA[1]) + (boxB[2]-boxB[0])*(boxB[3]-boxB[1]) - inter
    return inter / union if union > 0 else 0.0

def apply_nms(detections, iou_threshold=0.45):
    """Greedy NMS — keep highest-confidence box, suppress overlapping duplicates."""
    if not detections:
        return []
    dets = sorted(detections, key=lambda d: d["conf"], reverse=True)
    keep, used = [], set()
    for i, d in enumerate(dets):
        if i in used:
            continue
        keep.append(d)
        for j in range(i + 1, len(dets)):
            if j not in used and compute_iou(d["box"], dets[j]["box"]) > iou_threshold:
                used.add(j)
    return keep

def cluster_to_n(detections, n):
    """
    Reduce detections to at most n by merging spatially close duplicates.
    Any two boxes whose centers are within MAX_MATCH_DIST*0.6 px are the same pin;
    keep the highest-confidence one per cluster.
    """
    if len(detections) <= n:
        return detections
    dets = sorted(detections, key=lambda d: d["conf"], reverse=True)
    clusters, used = [], set()
    for i, d in enumerate(dets):
        if i in used:
            continue
        clusters.append(d)
        for j in range(i + 1, len(dets)):
            cx1, cy1 = get_center(d["box"])
            cx2, cy2 = get_center(dets[j]["box"])
            if j not in used and np.hypot(cx1-cx2, cy1-cy2) < MAX_MATCH_DIST * 0.6:
                used.add(j)
    return clusters[:n]

def match_detections_to_pins(detections, pin_boxes, pin_status):
    """
    Assign each detection to the nearest standing pin (centroid distance).
    Greedy: closest pair matched first, no pin or detection reused.
    Returns dict: pid → detection.
    """
    standing = {pid: box for pid, box in pin_boxes.items()
                if pin_status.get(pid) != "fallen"}
    pairs = []
    for pid, pbox in standing.items():
        px, py = get_center(pbox)
        for didx, det in enumerate(detections):
            dx, dy = get_center(det["box"])
            pairs.append((np.hypot(px-dx, py-dy), pid, didx))
    pairs.sort()

    matched, used_pid, used_det = {}, set(), set()
    for dist, pid, didx in pairs:
        if pid in used_pid or didx in used_det or dist > MAX_MATCH_DIST:
            continue
        matched[pid] = detections[didx]
        used_pid.add(pid)
        used_det.add(didx)
    return matched

def should_confirm_fall(window):
    """
    Two-path fall confirmation:
      Fast fall — CONSEC_THRESH consecutive fallen labels at end of window
      Slow fall — WINDOW_SIZE frames seen with WINDOW_RATIO fraction fallen
    """
    if not window:
        return False
    # Count consecutive fallen labels from the end
    consec = 0
    for label in reversed(window):
        if label == CLASS_FALLEN:
            consec += 1
        else:
            break
    if consec >= CONSEC_THRESH:
        return True
    if len(window) >= WINDOW_SIZE and window.count(CLASS_FALLEN) / len(window) >= WINDOW_RATIO:
        return True
    return False


# ── PIN STATE ─────────────────────────────────────────────────────────────────
pin_boxes        = {}   # pid → latest tracked box (last known position)
pin_fall_box     = {}   # pid → box frozen at moment of fall confirmation
pin_status       = {}   # pid → "standing" | "fallen"
pin_fall_time    = {}   # pid → timestamp (s) of confirmed fall
pin_fall_window  = {}   # pid → sliding window of recent class labels
pin_miss_counter = {}   # pid → consecutive frames with no matched detection
pin_initialized  = False
stable_buf       = []


# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(video_path)
out = cv2.VideoWriter(
    "/content/output_bowling.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps, (width, height)
)

frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    timestamp = frame_idx / fps
    annotated = frame.copy()

    # ── YOLO inference ────────────────────────────────────────────────────────
    results    = model(frame, conf=CONF_THRESHOLD, verbose=False)[0]
    detections = [
        {"box": box.tolist(), "class": int(cls), "conf": float(conf)}
        for box, cls, conf in zip(
            results.boxes.xyxy.cpu().numpy(),
            results.boxes.cls.cpu().numpy(),
            results.boxes.conf.cpu().numpy())
    ]
    detections = apply_nms(detections)

    # ── Initialization ────────────────────────────────────────────────────────
    if not pin_initialized:
        candidates = cluster_to_n(detections, NUM_PINS)
        stable_buf = stable_buf + [candidates] if len(candidates) == NUM_PINS else []

        if len(stable_buf) >= INIT_STABLE_FRAMES:
            for i, det in enumerate(sorted(stable_buf[-1], key=lambda d: get_center(d["box"])[0])):
                pid = i + 1
                pin_boxes[pid]        = det["box"]
                pin_status[pid]       = "standing"
                pin_fall_window[pid]  = []
                pin_miss_counter[pid] = 0
            pin_initialized = True
            centers = [(p, tuple(int(c) for c in get_center(b))) for p, b in pin_boxes.items()]
            print(f"✓ Initialized at {timestamp:.1f}s | pins: {centers}")

    # ── Tracking & fall detection ─────────────────────────────────────────────
    if pin_initialized:
        matched = match_detections_to_pins(detections, pin_boxes, pin_status)

        for pid in range(1, NUM_PINS + 1):
            if pin_status[pid] == "fallen":
                continue

            if pid in matched:
                det                   = matched[pid]
                pin_boxes[pid]        = det["box"]
                pin_miss_counter[pid] = 0

                pin_fall_window[pid].append(det["class"])
                if len(pin_fall_window[pid]) > WINDOW_SIZE:
                    pin_fall_window[pid].pop(0)

                if should_confirm_fall(pin_fall_window[pid]):
                    pin_status[pid]    = "fallen"
                    pin_fall_time[pid] = timestamp
                    pin_fall_box[pid]  = det["box"]
                    print(f"  Pin {pid} FALLEN at {timestamp:.1f}s  window={pin_fall_window[pid]}")

            else:
                pin_miss_counter[pid] += 1

                if pin_miss_counter[pid] == GONE_CONFIRM_FRAMES:
                    # Absent ~0.5s — fell and slid out of view; backdate timestamp
                    pin_status[pid]    = "fallen"
                    pin_fall_time[pid] = max(0.0, timestamp - GONE_CONFIRM_FRAMES / fps)
                    pin_fall_box[pid]  = pin_boxes[pid]
                    print(f"  Pin {pid} FALLEN (vanished) at {pin_fall_time[pid]:.1f}s")

                elif pin_miss_counter[pid] > FALL_MISS_TOLERANCE:
                    pin_fall_window[pid] = []

        # ── Auto-confirm last pin ──────────────────────────────────────────────
        standing = [p for p in range(1, NUM_PINS + 1) if pin_status[p] == "standing"]
        fallen   = [p for p in range(1, NUM_PINS + 1) if pin_status[p] == "fallen"]

        if len(standing) == 1 and len(fallen) == NUM_PINS - 1:
            last = standing[0]
            w    = pin_fall_window.get(last, [])
            if w.count(CLASS_FALLEN) >= 1:
                pin_status[last]    = "fallen"
                pin_fall_time[last] = timestamp
                pin_fall_box[last]  = pin_boxes.get(last, pin_boxes[last])
                print(f"  Pin {last} FALLEN (last pin) at {timestamp:.1f}s")

    # ── Render ────────────────────────────────────────────────────────────────
    if pin_initialized:
        for pid in range(1, NUM_PINS + 1):
            box = pin_fall_box.get(pid) if pin_status[pid] == "fallen" else pin_boxes.get(pid)
            if box is None:
                continue
            cx, cy = int(get_center(box)[0]), int(get_center(box)[1])

            if pin_status[pid] == "fallen":
                cv2.circle(annotated, (cx, cy), 35, (0, 0, 255), 3)
                cv2.putText(annotated, f"Pin {pid} fell @ {pin_fall_time[pid]:.1f}s",
                            (cx - 70, cy - 45), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 2)
            else:
                cv2.circle(annotated, (cx, cy), 35, (0, 255, 0), 2)
                cv2.putText(annotated, f"Pin {pid}",
                            (cx - 20, cy - 40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # ── HUD ───────────────────────────────────────────────────────────────────
    fallen_count = sum(1 for s in pin_status.values() if s == "fallen")
    cv2.putText(annotated, f"Pins down: {fallen_count}/{NUM_PINS}",
                (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 0), 2)
    cv2.putText(annotated, f"Time: {timestamp:.1f}s",
                (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 0), 2)

    # ── Final summary (last 2 seconds) ────────────────────────────────────────
    if timestamp >= total_duration - 2.0:
        for i, line in enumerate([
            "GAME OVER",
            f"Total Pins Knocked: {len(pin_fall_time)} / {NUM_PINS}",
            f"Duration: {total_duration:.1f}s"
        ]):
            cv2.putText(annotated, line,
                        (width // 2 - 220, height // 2 - 60 + i * 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)

    out.write(annotated)
    frame_idx += 1

cap.release()
out.release()
print(f"\nDone! Frames: {frame_idx}")
print(f"Fall times: {pin_fall_time}")

✓ Initialized at 0.1s | pins: [(1, (47, 548)), (2, (127, 551)), (3, (212, 544)), (4, (358, 554)), (5, (425, 547))]
  Pin 4 FALLEN (vanished) at 3.2s
  Pin 3 FALLEN (vanished) at 9.1s
  Pin 1 FALLEN (vanished) at 14.0s
  Pin 2 FALLEN (vanished) at 14.3s
  Pin 5 FALLEN (last pin) at 19.9s

Done! Frames: 769
Fall times: {4: 3.2029449475483283, 3: 9.141738704460854, 1: 13.979520135653642, 2: 14.279796224486297, 5: 19.88494988269587}


In [ ]:
from google.colab import files
files.download("/content/output_bowling.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>